In [1]:
import numpy as np
import pandas as pd
from numbers import Number
from IPython.display import display
import matplotlib.pyplot as plt
import re

In [2]:
# Data import and cleaning from previous milestone
base_url = "https://huggingface.co/datasets/google-research-datasets/go_emotions/resolve/main/simplified/"

train_df = pd.read_parquet(base_url + "train-00000-of-00001.parquet")
val_df = pd.read_parquet(base_url + "validation-00000-of-00001.parquet")
test_df = pd.read_parquet(base_url + "test-00000-of-00001.parquet")

# Label and group mappings 
LABEL_NAMES = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]
# Map each label to its corresponding group
GROUP_MAP = {
    "admiration":"positive","amusement":"positive","approval":"positive",
    "caring":"positive","desire":"positive","excitement":"positive",
    "gratitude":"positive","joy":"positive","love":"positive",
    "optimism":"positive","pride":"positive","relief":"positive",
    "anger":"negative","annoyance":"negative","disappointment":"negative",
    "disapproval":"negative","disgust":"negative","embarrassment":"negative",
    "fear":"negative","grief":"negative","nervousness":"negative",
    "remorse":"negative","sadness":"negative",
    "confusion":"ambiguous","curiosity":"ambiguous",
    "realization":"ambiguous","surprise":"ambiguous",
    "neutral":"neutral"
}
# Function to convert label IDs to their corresponding groups
def labels_to_groups_all(label_ids):
    names = [LABEL_NAMES[i] for i in label_ids]
    groups = [GROUP_MAP[n] for n in names]
    return list(dict.fromkeys(groups))

# Process each dataset split
def process_split(df):
    df = df.copy()
    df["all_groups"] = df["labels"].apply(labels_to_groups_all)
    df["num_distinct_groups"] = df["all_groups"].apply(len)
    clean = df[df["num_distinct_groups"] == 1].copy()
    clean["group_label"] = clean["all_groups"].apply(lambda g: g[0])
    conflict = df[df["num_distinct_groups"] > 1].copy()
    return clean, conflict

# Process all dataset splits
train_clean, train_conflict = process_split(train_df)
val_clean, val_conflict = process_split(val_df)
test_clean, test_conflict = process_split(test_df)

def clean_text(text):
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text) #remove hidden placeholder words
    text = re.sub(r"http\S+", "", text)               #remove links
    text = re.sub(r"\s+", " ", text).strip()          #remove extra whitespace   
    return text

train_clean["clean_text"] = train_clean["text"].apply(clean_text)
val_clean["clean_text"] = val_clean["text"].apply(clean_text)
test_clean["clean_text"] = test_clean["text"].apply(clean_text)

train_clean = train_clean[train_clean["clean_text"].str.strip() != ""]
val_clean = val_clean[val_clean["clean_text"].str.strip() != ""]
test_clean = test_clean[test_clean["clean_text"].str.strip() != ""]

# Remove duplicate text entries shared across splits to prevent data leakage
leak_texts = set(val_clean["clean_text"]) | set(test_clean["clean_text"])
train_clean = train_clean[~train_clean["clean_text"].isin(leak_texts)]


In [3]:
# Making a naive majority class predition benchmark
import time
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

naive_majority_classifier = DummyClassifier(strategy="most_frequent", random_state=42)

# Measure training runtime
t0 = time.perf_counter()
naive_majority_classifier.fit(train_clean["clean_text"], train_clean["group_label"])
naive_train_time_ms = (time.perf_counter() - t0) * 1000

# The naive majority classifier should just guess "positive" as the group label for every prediction

# Validation predictions evaluated on val_clean to protect the test set
t0 = time.perf_counter()
naive_val_preds = naive_majority_classifier.predict(val_clean["clean_text"])
naive_infer_time_ms = (time.perf_counter() - t0) * 1000
predictions = naive_val_preds

# Compute required metrics on val_clean
naive_val_f1 = f1_score(val_clean["group_label"], naive_val_preds, average="macro", zero_division=0)
naive_val_prec = precision_score(val_clean["group_label"], naive_val_preds, average="macro", zero_division=0)
naive_val_rec = recall_score(val_clean["group_label"], naive_val_preds, average="macro", zero_division=0)
naive_val_acc = accuracy_score(val_clean["group_label"], naive_val_preds)
accuracy = naive_val_acc

# Variables prepared for the Baseline Comparison Table
naive_eval_split = f"GoEmotions Validation Set (n={len(val_clean)})"
naive_time_str = f"{naive_train_time_ms:.2f} / {naive_infer_time_ms:.2f}"

print(f"Predictions: {predictions}")
print(f"Validation Accuracy: {naive_val_acc:.4f}")
print(f"Validation Macro F1 (Primary): {naive_val_f1:.4f}")
print(f"Validation Macro Precision: {naive_val_prec:.4f}")
print(f"Validation Macro Recall: {naive_val_rec:.4f}")
print(f"Training Time: {naive_train_time_ms:.2f} ms")
print(f"Inference Time: {naive_infer_time_ms:.2f} ms")


Predictions: ['positive' 'positive' 'positive' ... 'positive' 'positive' 'positive']
Validation Accuracy: 0.3878
Validation Macro F1 (Primary): 0.1397
Validation Macro Precision: 0.0970
Validation Macro Recall: 0.2500
Training Time: 8.12 ms
Inference Time: 0.24 ms


In [ ]:
# BASELINE COMPARISON TABLE
import pandas as pd

# Check if Naive Bayes model/predictions/metrics exist, otherwise use clean placeholders
if "nb_val_f1" not in globals() and "nb_val_preds" in globals():
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    nb_val_f1 = f1_score(val_clean["group_label"], nb_val_preds, average="macro", zero_division=0)
    nb_val_prec = precision_score(val_clean["group_label"], nb_val_preds, average="macro", zero_division=0)
    nb_val_rec = recall_score(val_clean["group_label"], nb_val_preds, average="macro", zero_division=0)
    nb_val_acc = accuracy_score(val_clean["group_label"], nb_val_preds)

nb_f1 = round(nb_val_f1, 4) if "nb_val_f1" in globals() else "[Pending Matthew W.]"
nb_prec = round(nb_val_prec, 4) if "nb_val_prec" in globals() else "[Pending Matthew W.]"
nb_rec = round(nb_val_rec, 4) if "nb_val_rec" in globals() else "[Pending Matthew W.]"
nb_acc = round(nb_val_acc, 4) if "nb_val_acc" in globals() else "[Pending Matthew W.]"
nb_time = (
    f"{nb_train_time_ms:.2f} / {nb_infer_time_ms:.2f}"
    if ("nb_train_time_ms" in globals() and "nb_infer_time_ms" in globals())
    else "[Pending Matthew W.]"
)
nb_notes = globals().get(
    "nb_settings_notes",
    "[Pending Matthew W.]"
)

# Comparison table
baseline_table_data = [
    {
        "Model / Baseline Tier": "Naive Benchmark (DummyClassifier: most_frequent)",
        "Evaluation Split": naive_eval_split,
        "Macro F1 (Primary)": round(naive_val_f1, 4),
        "Macro Precision": round(naive_val_prec, 4),
        "Macro Recall": round(naive_val_rec, 4),
        "Accuracy": round(naive_val_acc, 4),
        "Train / Infer Time (ms)": naive_time_str,
        "Key Settings / Notes": "strategy='most_frequent', random_state=42 (always predicts 'positive')"
    },
    {
        "Model / Baseline Tier": "Course Baseline (Naive Bayes)",
        "Evaluation Split": f"GoEmotions Validation Set (n={len(val_clean)})",
        "Macro F1 (Primary)": nb_f1,
        "Macro Precision": nb_prec,
        "Macro Recall": nb_rec,
        "Accuracy": nb_acc,
        "Train / Infer Time (ms)": nb_time,
        "Key Settings / Notes": nb_notes
    },
    {
        "Model / Baseline Tier": "Reference: BERT-base (Demszky et al., 2020)",
        "Evaluation Split": "GoEmotions Test (n=5,427)",
        "Macro F1 (Primary)": 0.69,
        "Macro Precision": 0.65,
        "Macro Recall": 0.74,
        "Accuracy": "N/A",
        "Train / Infer Time (ms)": "N/A",
        "Key Settings / Notes": "BERT-base baseline values from Demszky et al. (2020, Table 5)"
    }
]

baseline_comparison_df = pd.DataFrame(baseline_table_data)

# Table formatting
pd.set_option("display.max_colwidth", None)
metric_columns = [
    "Macro F1 (Primary)",
    "Macro Precision",
    "Macro Recall",
    "Accuracy",
]
numeric_formatters = {
    column: lambda value: f"{value:g}" if isinstance(value, Number) else value
    for column in metric_columns
}
baseline_comparison_style = (
    baseline_comparison_df.style
    .format(numeric_formatters)
    .set_properties(**{
        "white-space": "normal",
        "overflow-wrap": "anywhere",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [("white-space", "normal"), ("overflow-wrap", "anywhere")],
        },
        {
            "selector": "td",
            "props": [("white-space", "normal"), ("overflow-wrap", "anywhere")],
        },
        {
            "selector": "table",
            "props": [("table-layout", "auto"), ("width", "100%")],
        },
    ])
)
display(baseline_comparison_style)


,Model / Baseline Tier,Evaluation Split,Macro F1 (Primary),Macro Precision,Macro Recall,Accuracy,Train / Infer Time (ms),Key Settings / Notes
0,Naive Benchmark (DummyClassifier: most_frequent),GoEmotions Validation Set (n=5005),0.1397,0.097,0.25,0.3878,8.12 / 0.24,"strategy='most_frequent', random_state=42 (always predicts 'positive')"
1,Course Baseline (Naive Bayes),GoEmotions Validation Set (n=5005),[Pending Matthew W.],[Pending Matthew W.],[Pending Matthew W.],[Pending Matthew W.],[Pending Matthew W.],[Pending Matthew W.]
2,"Reference: BERT-base (Demszky et al., 2020)","GoEmotions Test (n=5,427)",0.69,0.65,0.74,N/A,N/A,"BERT-base baseline values from Demszky et al. (2020, Table 5)"
